# Field Mapping: Jefferson Farm 2026

Load field boundaries from KMZ and overlay USDA SSURGO soil data.

In [ ]:
import sys
sys.path.insert(0, '../skills/my-farm-advisor/soil/ssurgo-soil/src')

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import time

# Load field boundaries from KMZ
kmz_path = '../skills/my-farm-advisor/field-management/field-boundaries/2026 Jefferson Farm Fields.kmz'
fields = gpd.read_file(kmz_path)

print(f"Loaded {len(fields)} fields")
print(f"CRS: {fields.crs}")
print(f"Columns: {list(fields.columns)}")
fields.head()

In [ ]:
# Ensure CRS is WGS84 (EPSG:4326) for NRCS API queries
if fields.crs is None:
    fields = fields.set_crs('EPSG:4326')
elif fields.crs != 'EPSG:4326':
    fields = fields.to_crs('EPSG:4326')

print(f"Fields CRS: {fields.crs}")
print(f"\nAll {len(fields)} field names:")
for name in fields['Name'].values:
    print(f"  - {name}")

In [ ]:
# Quick plot of field boundaries
fig, ax = plt.subplots(figsize=(12, 10))
fields.plot(ax=ax, edgecolor='black', facecolor='lightblue', alpha=0.7)
ax.set_title('Jefferson Farm Field Boundaries - 2026')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## Fetch Soil Data from NRCS SSURGO API

In [ ]:
from ssurgo_soil import get_soil_at_point

# Query soil data for each field using centroid point
# Note: NRCS API has rate limits; adding delay between requests
print("Fetching soil data from NRCS SSURGO API...")
print("(This may take a few minutes for 57 fields)\n")

soil_results = []

for idx, row in fields.iterrows():
    field_name = row['Name']
    centroid = row.geometry.centroid
    
    try:
        soil = get_soil_at_point(lon=centroid.x, lat=centroid.y, max_depth_cm=30)
        
        if not soil.empty:
            # Get the top horizon (shallowest depth)
            soil_sorted = soil.sort_values('hzdept_r').head(1)
            soil_sorted = soil_sorted.copy()
            soil_sorted['field_id'] = field_name
            soil_results.append(soil_sorted)
            print(f"✓ {field_name}: OM={soil_sorted['om_r'].values[0]}, pH={soil_sorted['ph1to1h2o_r'].values[0]}")
        else:
            print(f"✗ {field_name}: No soil data")
    except Exception as e:
        print(f"✗ {field_name}: Error - {str(e)[:50]}")
    
    # Rate limiting - small delay between requests
    time.sleep(0.3)

print(f"\nCompleted: {len(soil_results)} fields with soil data")

In [ ]:
# Combine results
if soil_results:
    soil_data = pd.concat(soil_results, ignore_index=True)
    print(f"Soil data records: {len(soil_data)}")
    print(f"\nColumns available: {list(soil_data.columns)}")
    
    # Rename hzdept_r to indicate it's depth in cm (convert from mm if needed)
    # The API returns values that need scaling
    soil_data['depth_cm'] = soil_data['hzdept_r'].apply(lambda x: x / 10 if pd.notna(x) and x > 100 else x)
    
    soil_data[['field_id', 'om_r', 'ph1to1h2o_r', 'drainagecl', 'awc_r']].head(10)

In [ ]:
# Summary statistics for key soil properties
print("Soil Property Summary (Top 30cm):")
print(f"  Organic Matter: {soil_data['om_r'].mean():.1f}% (range: {soil_data['om_r'].min():.1f} - {soil_data['om_r'].max():.1f}%)")
print(f"  pH: {soil_data['ph1to1h2o_r'].mean():.1f} (range: {soil_data['ph1to1h2o_r'].min():.1f} - {soil_data['ph1to1h2o_r'].max():.1f})")

if 'drainagecl' in soil_data.columns:
    print(f"\nDrainage Class Distribution:")
    print(soil_data['drainagecl'].value_counts())

## Overlay Soil Data on Field Boundaries

In [ ]:
# Merge soil data with field boundaries
fields_with_soil = fields.merge(soil_data, left_on='Name', right_on='field_id', how='left')

print(f"Fields with soil data: {fields_with_soil['om_r'].notna().sum()} / {len(fields_with_soil)}")
fields_with_soil[['Name', 'om_r', 'ph1to1h2o_r', 'drainagecl']].head(10)

In [ ]:
# Plot: pH by Field
fig, ax = plt.subplots(figsize=(14, 10))
fields_with_soil.plot(column='ph1to1h2o_r', ax=ax, legend=True,
                       cmap='RdYlGn', edgecolor='black', linewidth=0.5,
                       legend_kwds={'label': 'pH', 'shrink': 0.6})
ax.set_title('Jefferson Farm - Soil pH by Field', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Organic Matter by Field (scaled - the API returns values in different units)
# Note: Values may need interpretation based on local soil conditions
fig, ax = plt.subplots(figsize=(14, 10))
fields_with_soil.plot(column='om_r', ax=ax, legend=True, 
                       cmap='YlGn', edgecolor='black', linewidth=0.5,
                       legend_kwds={'label': 'Organic Matter (%)', 'shrink': 0.6})
ax.set_title('Jefferson Farm - Organic Matter by Field', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Available Water Capacity by Field
fig, ax = plt.subplots(figsize=(14, 10))
fields_with_soil.plot(column='awc_r', ax=ax, legend=True,
                       cmap='Blues', edgecolor='black', linewidth=0.5,
                       legend_kwds={'label': 'AWC (in/in)', 'shrink': 0.6})
ax.set_title('Jefferson Farm - Available Water Capacity by Field', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## Save Output Data

In [ ]:
# Save field boundaries with soil data as GeoJSON
output_geojson = 'jefferson_farm_soil_overlay.geojson'
fields_with_soil.to_file(output_geojson, driver='GeoJSON')
print(f"Saved: {output_geojson}")

# Save soil data as CSV
output_csv = 'jefferson_farm_soil_data.csv'
soil_data.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

## Summary Table

In [ ]:
# Display summary table of fields with soil properties
summary = fields_with_soil[['Name', 'om_r', 'ph1to1h2o_r', 'drainagecl', 'awc_r', 'cec7_r']].copy()
summary.columns = ['Field', 'OM (%)', 'pH', 'Drainage', 'AWC (in/in)', 'CEC (meq/100g)']
summary = summary.sort_values('Field')
pd.set_option('display.max_rows', 60)
summary

## Notes

- Soil data sourced from USDA NRCS Soil Data Access (SDA) API
- Field boundaries loaded from KMZ file: `2026 Jefferson Farm Fields.kmz`
- 57 total fields in Jefferson Farm
- Some fields may show no data due to API query limitations or no SSURGO coverage